# Preparing Pork Supply and Demand Data

This notebook loads USDA NASS annual corn price data (1866-2025), adjusts prices for inflation using the Consumer Price Index (CPI), and prepares the data for predictive modeling.

## Data Loading

Import libraries and load the corn price and CPI datasets from the project's GitHub repository.

In [30]:
import pandas as pd
import numpy as np
import requests
from dotenv import load_dotenv
import sys 
from pathlib import Path
import os

# ------------------------------ #
# Must run together to access NASS API key
# ------------------------------ #
src_path = Path.cwd() / 'src'
sys.path.insert(0, str(src_path))
from utils.nass_api import NASS_API_KEY
from data.nass_utils import clean_nass_value
# ------------------------------ #

### Load CPI Data

In [31]:
# Load CPI data directly from GitHub URL
cpi = pd.read_csv('data/' + 'cpi_1913_2024.csv')

## Prepare Annual Pork Production Data

Pulls pork production data from the USDA NASS API, filters to exclude calves, cleans the `Value` column, and saves the result to CSV.

In [32]:
base_url = "https://quickstats.nass.usda.gov/api/api_GET/"

params = {'key': NASS_API_KEY,
          'source_desc': 'SURVEY',
          'sector_desc': 'ANIMALS & PRODUCTS',
          'group_desc': 'LIVESTOCK',
          'commodity_desc': 'HOGS',
          'statisticcat_desc': 'PRODUCTION',
          'short_desc': 'HOGS - PRODUCTION, MEASURED IN LB',
          'agg_level_desc': 'NATIONAL'
         }

response = requests.get(base_url, params=params)

hog_production = response.json()

hog_production = pd.DataFrame(hog_production['data'])

In [33]:
hog_production_clean = clean_nass_value(hog_production, 'Value')
sel_columns = ['year', 'Value']
hog_production_clean = hog_production_clean[sel_columns].copy()
hog_production_clean = hog_production_clean.rename(columns={'Value': 'hog_production_lb'})
hog_production_clean['year'] = hog_production_clean['year'].astype(int)
hog_production_clean['hog_production_lb'] = hog_production_clean['hog_production_lb'] / 1_000_000_000
hog_production_clean.head()

,year,hog_production_lb
0,2024,41.604752
1,2023,42.424619
2,2022,40.674251
3,2021,39.992279
4,2020,40.428021


In [34]:
base_url = "https://quickstats.nass.usda.gov/api/api_GET/"

params = {'key': NASS_API_KEY,
          'source_desc': 'SURVEY',
          'sector_desc': 'ANIMALS & PRODUCTS',
          'group_desc': 'LIVESTOCK',
          'commodity_desc': 'HOGS',
          'statisticcat_desc': 'PRICE RECEIVED',
          'short_desc': 'HOGS - PRICE RECEIVED, MEASURED IN $ / CWT',
          'agg_level_desc': 'NATIONAL'
         }

response = requests.get(base_url, params=params)

hog_prices = response.json()

hog_prices = pd.DataFrame(hog_prices['data'])

In [35]:
hog_prices_yearly = hog_prices[hog_prices['reference_period_desc'] == 'YEAR'].copy()
hog_prices_yearly_clean = clean_nass_value(hog_prices_yearly, 'Value')
sel_columns = ['year', 'Value']
hog_prices_yearly_clean = hog_prices_yearly_clean[sel_columns].copy()
hog_prices_yearly_clean = hog_prices_yearly_clean.rename(columns={'Value': 'hog_prices_cwt'})
hog_prices_yearly_clean['year'] = hog_prices_yearly_clean['year'].astype(int)
hog_prices_yearly_clean['hog_prices_lb'] = hog_prices_yearly_clean['hog_prices_cwt'] / 100
hog_prices_yearly_clean = hog_prices_yearly_clean.drop(columns=['hog_prices_cwt'])
hog_prices_yearly_clean.head()

,year,hog_prices_lb
16,2024,0.641
30,2023,0.620
44,2022,0.730
58,2021,0.685
72,2020,0.471


In [36]:
base_url = "https://quickstats.nass.usda.gov/api/api_GET/"

params = {'key': NASS_API_KEY,
          'source_desc': 'SURVEY',
          'sector_desc': 'CROPS',
          'group_desc': 'FIELD CROPS',
          'commodity_desc': 'CORN',
          'statisticcat_desc': 'PRICE RECEIVED',
          'short_desc': 'CORN, GRAIN - PRICE RECEIVED, MEASURED IN $ / BU',
          'agg_level_desc': 'NATIONAL'
         }

response = requests.get(base_url, params=params)

corn_prices = response.json()

corn_prices = pd.DataFrame(corn_prices['data'])

In [37]:
corn_prices_yearly = corn_prices[corn_prices['reference_period_desc'] == 'MARKETING YEAR'].copy()
corn_prices_yearly_clean = clean_nass_value(corn_prices_yearly, 'Value')
sel_columns = ['year', 'Value']
corn_prices_yearly_clean = corn_prices_yearly_clean[sel_columns].copy()
corn_prices_yearly_clean = corn_prices_yearly_clean.rename(columns={'Value': 'corn_prices_bu'})
corn_prices_yearly_clean['year'] = corn_prices_yearly_clean['year'].astype(int)
corn_prices_yearly_clean.head()

,year,corn_prices_bu
2,2025,4.10
15,2024,4.24
29,2023,4.55
43,2022,6.54
57,2021,6.00


### Soybean Meal Prices

Retrieved from [USDA ERS Oil Crops Yearbook](https://www.ers.usda.gov/data-products/oil-crops-yearbook)

In [38]:
# Path to ERS OilCrops Excel file (relative to project root)
file_path = 'data/raw/OilCropsAllTables.xlsx'

# Row 0 is the table title; row 1 contains column headers → header=1
# usecols selects only the marketing year (col 0) and price (col 9)
df_soy = pd.read_excel(
    file_path,
    sheet_name='Table04',
    header=1,
    usecols=[0, 9]
)

# Rename to clean column names
df_soy.columns = ['marketing_year', 'soybean_meal_price']

In [39]:

# Drop rows with no price data (trailing empty rows)
df_soy = df_soy.dropna(subset=['soybean_meal_price']).copy()

# Extract start year from marketing year string (e.g., "1980/81" → 1980)
df_soy['year'] = df_soy['marketing_year'].str[:4].astype(int)

# Keep only year and price columns
soybean_meal_prices = df_soy[['year', 'soybean_meal_price']].reset_index(drop=True)
soybean_meal_prices['soybean_meal_price_lb'] = soybean_meal_prices['soybean_meal_price'] / 2000
soybean_meal_prices = soybean_meal_prices.rename(columns={'soybean_meal_price': 'soybean_meal_price_short_ton'})
soybean_meal_prices = soybean_meal_prices.drop(columns=['soybean_meal_price_short_ton'])

print(f"Rows: {len(soybean_meal_prices)}")
print(f"Year range: {soybean_meal_prices['year'].min()} – {soybean_meal_prices['year'].max()}")
soybean_meal_prices.head()

Rows: 46
Year range: 1980 – 2025


,year,soybean_meal_price_lb
0,1980,0.117565
1,1981,0.098310
2,1982,0.100470
3,1983,0.101605
4,1984,0.068200


### Join Datasets

In [40]:
from functools import reduce

# Join all datasets on year (inner join keeps only overlapping years)
dfs = [hog_production_clean, hog_prices_yearly_clean, corn_prices_yearly_clean, soybean_meal_prices, cpi]
clean_df = reduce(lambda left, right: left.merge(right, on='year', how='inner'), dfs)

# Get 2020 CPI base value
cpi_2020 = clean_df.loc[clean_df['year'] == 2020, 'annual_avg_cpi'].values[0]

# Define which columns to inflation-adjust
price_cols = ['hog_prices_lb', 'corn_prices_bu', 'soybean_meal_price_lb']  # adjust names to match your dfs

# Convert all price columns to real (2020 dollars) in one pass
for col in price_cols:
    clean_df[f'{col}'] = (clean_df[col] / clean_df['annual_avg_cpi']) * cpi_2020

clean_df = clean_df.drop(columns=['annual_avg_cpi'])

In [41]:
clean_df.head()

,year,hog_production_lb,hog_prices_lb,corn_prices_bu,soybean_meal_price_lb
0,2024,41.604752,0.528861,3.498238,0.123664
1,2023,42.424619,0.526622,3.864727,0.163130
2,2022,40.674251,0.645579,5.783684,0.199825
3,2021,39.992279,0.654263,5.730767,0.210037
4,2020,40.428021,0.471000,4.530000,0.196155


In [42]:
clean_df.to_csv('./data/pork_supply_data.csv', index=False)